# Atelier Scikit-learn — Prédiction de l'état d'un capteur IoT

**Contexte.** Des capteurs mesurent température, humidité, pression et consommation. Chaque mesure a un **état** : `OK`, `ALERTE` ou `ERREUR`. On veut un modèle qui **prédit automatiquement l'état** d'une mesure à partir de ses valeurs numériques.

**Workflow ML classique suivi :**
Dataset → Chargement → Exploration → Nettoyage → X / y → Train/Test → Prétraitement → Modèle → `fit()` → `predict()` → Évaluation → Sauvegarde → Chargement → Réutilisation.

## Partie 0 — Mise en place de l'environnement

On importe **pandas** (manipulation de données), **matplotlib** et **seaborn** (visualisation). Le CSV se trouve dans `data/` ; le notebook s'exécutant depuis `notebooks/`, on remonte d'un niveau (`../data/...`).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/mesures_capteurs.csv")
print("Forme du dataframe :", df.shape)
df.head()

Forme du dataframe : (605, 9)


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


### Exploration du dataframe

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    str    
 1   date_heure    605 non-null    str    
 2   id_capteur    605 non-null    str    
 3   batiment      605 non-null    str    
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    str    
dtypes: float64(4), str(5)
memory usage: 42.7 KB


In [3]:
df.describe()

,temperature,humidite,pression,consommation
count,599.000000,600.00000,600.000000,600.000000
mean,24.878314,64.92620,1012.221900,208.675417
std,4.059576,10.76905,10.599042,72.243567
min,-18.500000,28.52000,850.000000,18.120000
25%,22.570000,58.17250,1006.790000,160.177500
50%,24.860000,65.37500,1012.855000,206.150000
75%,27.275000,71.61500,1017.827500,254.127500
max,58.700000,145.00000,1038.430000,875.000000


In [4]:
# Repartition de la variable cible 'etat'
df["etat"].value_counts(dropna=False)

etat
OK        567
ALERTE     29
ERREUR      5
NaN         4
Name: count, dtype: int64

**Observation importante.** Les classes sont **très déséquilibrées** : la grande majorité des mesures sont `OK`, `ALERTE` est rare et `ERREUR` très rare. Ce déséquilibre aura des conséquences sur l'évaluation (Partie 7).